
# Startup Profit Prediction Analysis

This notebook performs an exploratory data analysis, data preprocessing, feature engineering, and checks for multicollinearity in a dataset containing information about 50 startups.

In [ ]:
import numpy as np
import pandas as pd       #Import pandas for data manipulation
import matplotlib.pyplot as plt            # Import numpy for numerical operations


## 1. Data Loading

Here, the dataset `50_Startups.csv` is loaded into a pandas DataFrame named `data`.

In [ ]:
data = pd.read_csv("/content/50_Startups (1).csv")
data

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94
5,131876.90,99814.71,362861.36,New York,156991.12
6,134615.46,147198.87,127716.82,California,156122.51
7,130298.13,145530.06,323876.68,Florida,155752.60
8,120542.52,148718.95,311613.29,New York,152211.77
9,123334.88,108679.17,304981.62,California,149759.96


## 2. Initial Data Exploration and Cleaning

This section involves a preliminary look at the data, checking for missing values, inspecting column names and data types, and handling duplicate entries.

### 2.1 Inspecting Data Overview

In [ ]:
data.head()     # Display the first 5 rows of the DataFrame

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


#### Check for Missing Values

Calculate the percentage of missing values for each column to understand data completeness.

In [ ]:
data.isna()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,False,False
6,False,False,False,False,False
7,False,False,False,False,False
8,False,False,False,False,False
9,False,False,False,False,False


In [ ]:
data.isna().sum()/100

,0
R&D Spend,0.0
Administration,0.0
Marketing Spend,0.0
State,0.0
Profit,0.0


In [ ]:
# for introducing NAN values
# data.iloc[0:10, :] = np.nan
# data.iloc[0:10, [0,1]] = np.nan

In [ ]:
# If we have null values, then check 70% criteria (and we got 70% then remove that particular column)
data.isna().sum()/len(data)*100       # Calculate percentage of NA values per column

,0
R&D Spend,0.0
Administration,0.0
Marketing Spend,0.0
State,0.0
Profit,0.0


In [ ]:
data.loc[:,'R&D Spend'] == ' '

,R&D Spend
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


#### Inspect Column Names and Data Types

Review column names and data types, and the overall dimensions of the dataset.

In [ ]:
data.columns      # Display column names

Index(['R&D Spend', 'Administration', 'Marketing Spend', 'State', 'Profit'], dtype='object')

In [ ]:
# for removal of columns(If we find >70% null values)
#data_cleaned = data[['R&D Spend', 'Administration']]
#data_cleaned.head()

In [ ]:
# Introducing some duplicates
# data.iloc[0:10,:] = data.iloc[1,:]           (from 0 to 9 rows)

# Remove duplicate rows from the DataFrame. The default for drop_duplicates is axis=0 (rows).

In [ ]:
# drop duplicates in rows
data = data.drop_duplicates()
print(f"Shape of data after dropping duplicate rows: {data.shape}")

Shape of data after dropping duplicate rows: (50, 5)


In [ ]:
# Display the shape of the DataFrame after duplicate row removal.
data.shape

(50, 5)

## 3. Feature Engineering: One-Hot Encoding for 'State' Column

The 'State' column, being a categorical variable, needs to be converted into a numerical format for most machine learning models. One-hot encoding is used for this transformation, creating new binary columns for each unique state.

`pd.get_dummies` is utilized, and `drop_first=False` is implicitly used here. One of the dummy variables will be explicitly dropped later during VIF calculation to avoid perfect multicollinearity with the intercept (the 'dummy variable trap').

**Note:** `data_encoded` is created from the `data` DataFrame *after* duplicate rows have been handled.

In [ ]:
# Perform one-hot encoding on the 'State' column, creating new binary columns.
data_encoded = pd.get_dummies(data, columns=['State'])
# Display the first few rows of the DataFrame with one-hot encoded 'State' columns to verify.
display(data_encoded.head())

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,False,True
1,162597.70,151377.59,443898.53,191792.06,True,False,False
2,153441.51,101145.55,407934.54,191050.39,False,True,False
3,144372.41,118671.85,383199.62,182901.99,False,False,True
4,142107.34,91391.77,366168.42,166187.94,False,True,False


One Hot Encoding

In [ ]:
data_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   R&D Spend         50 non-null     float64
 1   Administration    50 non-null     float64
 2   Marketing Spend   50 non-null     float64
 3   Profit            50 non-null     float64
 4   State_California  50 non-null     bool   
 5   State_Florida     50 non-null     bool   
 6   State_New York    50 non-null     bool   
dtypes: bool(3), float64(4)
memory usage: 1.8 KB


In [ ]:
# OR
# categories = data['State'].unique()

In [ ]:
# for i in categories:
#   data[i] = np.where(data['State'] == i,1,0)     # we also use get_dumies for one hot encoding

In [ ]:
# data.head()

,R&D Spend,Administration,Marketing Spend,State,Profit,New York,California,Florida
0,165349.20,136897.80,471784.10,New York,192261.83,1,0,0
1,162597.70,151377.59,443898.53,California,191792.06,0,1,0
2,153441.51,101145.55,407934.54,Florida,191050.39,0,0,1
3,144372.41,118671.85,383199.62,New York,182901.99,1,0,0
4,142107.34,91391.77,366168.42,Florida,166187.94,0,0,1


## 4. Correlation Analysis

This section examines the relationships between variables, particularly focusing on how different features correlate with 'Profit' and with each other. A correlation matrix provides an overview of these relationships.


In [ ]:
data.info() # Display concise summary of the DataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


# Calculate and display the correlation matrix for all columns in the encoded DataFrame.
# This helps in understanding linear relationships between variables.

In [ ]:
data_encoded.corr()

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
R&D Spend,1.000000,0.241955,0.724248,0.972900,-0.143165,0.105711,0.039068
Administration,0.241955,1.000000,-0.032154,0.200717,-0.015478,0.010493,0.005145
Marketing Spend,0.724248,-0.032154,1.000000,0.747766,-0.168875,0.205685,-0.033670
Profit,0.972900,0.200717,0.747766,1.000000,-0.145837,0.116244,0.031368
State_California,-0.143165,-0.015478,-0.168875,-0.145837,1.000000,-0.492366,-0.515152
State_Florida,0.105711,0.010493,0.205685,0.116244,-0.492366,1.000000,-0.492366
State_New York,0.039068,0.005145,-0.033670,0.031368,-0.515152,-0.492366,1.000000


#### Individual Correlations with 'Profit'

Calculate the correlation coefficient between 'Profit' and each individual feature to see direct linear relationships.

In [ ]:
# Display the columns of the `data_encoded` DataFrame for reference.
data_encoded.columns

Index(['R&D Spend', 'Administration', 'Marketing Spend', 'Profit',
       'State_California', 'State_Florida', 'State_New York'],
      dtype='object')

In [ ]:
# Calculate the Pearson correlation coefficient between 'Profit' and 'R&D Spend'.
# This provides a direct measure of their linear relationship.
np.corrcoef(data_encoded['Profit'].values, data_encoded['R&D Spend'].values)

array([[1.        , 0.97290047],
       [0.97290047, 1.        ]])

In [ ]:
# Loop through all columns in `data_encoded` to print the correlation between 'Profit' and each feature.
for col_name in data_encoded.columns:
  # Ensure 'Profit' is not correlated with itself and avoid unnecessary computations if col_name is already 'Profit'.
  if col_name != 'Profit':
    print(f"Correlation between Profit and {col_name} =\n{np.corrcoef(data_encoded['Profit'], data_encoded[col_name])}\n")

Correlation between Profit and R&D Spend =
[[1.         0.97290047]
 [0.97290047 1.        ]]

Correlation between Profit and Administration =
[[1.         0.20071657]
 [0.20071657 1.        ]]

Correlation between Profit and Marketing Spend =
[[1.         0.74776572]
 [0.74776572 1.        ]]

Correlation between Profit and State_California =
[[ 1.         -0.14583704]
 [-0.14583704  1.        ]]

Correlation between Profit and State_Florida =
[[1.         0.11624426]
 [0.11624426 1.        ]]

Correlation between Profit and State_New York =
[[1.        0.0313676]
 [0.0313676 1.       ]]



## 5. Multicollinearity Check: Variance Inflation Factor (VIF)

Multicollinearity, a high correlation between independent variables, can negatively affect the reliability and interpretation of regression model coefficients. VIF is calculated to detect and measure the strength of this correlation. Variables with high VIF (typically greater than 5 or 10) may indicate a multicollinearity issue.

To handle the 'dummy variable trap' (perfect multicollinearity that occurs when including all dummy variables for a categorical feature along with an intercept term), one of the one-hot encoded state columns (specifically `State_New York` in this case) is dropped before calculating VIF. This ensures that the model matrix is not linearly dependent.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant



In [ ]:
# Define independent variables (X) by dropping the dependent variable 'Profit'.
X = data_encoded.drop('Profit', axis=1)



In [ ]:
# Convert any boolean columns (from one-hot encoding) to integer (0 or 1) as VIF requires numerical input.
for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)

# Drop one of the one-hot encoded 'State' columns to avoid perfect multicollinearity (dummy variable trap).
# 'State_New York' is chosen here as the reference category to be dropped.
X = X.drop('State_New York', axis=1)

# Add a constant (intercept) term to the DataFrame, as VIF calculation typically includes it.
X_const = add_constant(X, has_constant='add') # Use has_constant='add' to prevent adding constant if it already exists



In [ ]:
# Prepare an empty DataFrame to store VIF results.
vif_data = pd.DataFrame()
vif_data['feature'] = X_const.columns # Store feature names
# Calculate VIF for each independent variable using a list comprehension.
vif_data['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("VIF for Independent Variables:")
display(vif_data.round(2)) # Display VIF values rounded to two decimal places.

VIF for Independent Variables:


,feature,VIF
0,const,27.13
1,R&D Spend,2.50
2,Administration,1.18
3,Marketing Spend,2.42
4,State_California,1.34
5,State_Florida,1.36


In [ ]:
X = data_encoded.loc[:,['R&D Spend', 'Administration', 'Marketing Spend', 'State_California', 'State_Florida']]
Y = data_encoded.loc[:,'Profit']

In [ ]:
X# = data_encoded.iloc[:,[0,1,2,-3,-2,-1]].values
#Y = data_encoded.iloc[:,-4].values

In [ ]:
X.head()

,R&D Spend,Administration,Marketing Spend,State_California,State_Florida
0,165349.20,136897.80,471784.10,False,False
1,162597.70,151377.59,443898.53,True,False
2,153441.51,101145.55,407934.54,False,True
3,144372.41,118671.85,383199.62,False,False
4,142107.34,91391.77,366168.42,False,True


In [ ]:
Y.head()

,Profit
0,192261.83
1,191792.06
2,191050.39
3,182901.99
4,166187.94


In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X,Y)

LinearRegression()

In [ ]:
y_pred = regressor.predict(X)
y_pred

array([192390.57136321, 189071.32010493, 182276.18673284, 173584.97618975,
       172277.13381667, 163473.80711841, 158099.29278417, 160155.64464939,
       151634.74332631, 154829.66252709, 135664.64259063, 135528.6007816 ,
       129282.91780691, 127431.24898632, 149694.3827755 , 146143.63551378,
       116854.0745287 , 130085.40993358, 129149.72574274, 115594.18840731,
       116570.73444001, 117201.50508974, 114833.30513844, 110123.79610587,
       113294.37345154, 102200.26891867, 110765.30116879, 114279.80403011,
       101818.58738692, 101721.04202941,  99629.0105373 ,  97617.29632021,
        98988.23660304,  98061.35894838,  88974.70416295,  90420.0096061 ,
        75423.0928656 ,  89577.70222195,  69606.52160903,  83684.97603991,
        74762.74617648,  74956.31104908,  70575.99371382,  60100.2682187 ,
        64585.14721292,  47588.36470994,  56272.99268496,  46468.23200485,
        49123.07308286,  48185.0387926 ])

In [ ]:
regressor.intercept_

np.float64(50083.456812503464)

In [ ]:
regressor.coef_

array([ 8.06023114e-01, -2.70043196e-02,  2.69798610e-02,  4.18870191e+01,
        2.40675812e+02])

In [ ]:
print(X.shape)

(50, 5)


In [ ]:
print(regressor.coef_.shape)

(5,)


In [ ]:
print(regressor.intercept_.shape)

()


In [ ]:
np.dot(X,regressor.coef_)

array([142307.1145507045, 138987.8632924304, 132192.72992033718,
       123501.5193772468, 122193.67700416542, 113390.35030590641,
       108015.8359716701, 110072.18783688251, 101551.28651380472,
       104746.20571458185, 85581.18577812916, 85445.1439690944,
       79199.46099440413, 77347.79217381855, 99610.92596299607,
       96060.17870128015, 66770.61771619374, 80001.95312108105,
       79066.26893023195, 65510.73159480572, 66487.27762750805,
       67118.04827723956, 64749.848325934, 60040.33929336701,
       63210.91663904114, 52116.81210616648, 60681.84435628951,
       64196.34721760599, 51735.13057441451, 51637.5852169065,
       49545.55372480058, 47533.83950770194, 48904.77979053655,
       47977.90213587761, 38891.2473504446, 40336.5527935932,
       25339.636053095255, 39494.24540945023, 19523.064796526334,
       33601.519227402205, 24679.289363972057, 24872.85423657223,
       20492.53690131644, 10016.811406198418, 14501.690400420446,
       -2495.0921025634057, 6189.5

## Matrix Multiplication

In [ ]:
np.dot(X,regressor.coef_)+regressor.intercept_

array([192390.57136320794, 189071.32010493387, 182276.18673284064,
       173584.97618975028, 172277.13381666888, 163473.80711840987,
       158099.29278417357, 160155.64464938597, 151634.74332630818,
       154829.6625270853, 135664.64259063263, 135528.60078159787,
       129282.91780690759, 127431.24898632203, 149694.38277549954,
       146143.6355137836, 116854.07452869721, 130085.40993358451,
       129149.7257427354, 115594.18840730918, 116570.73444001153,
       117201.50508974303, 114833.30513843746, 110123.79610587048,
       113294.37345154461, 102200.26891866994, 110765.30116879297,
       114279.80403010946, 101818.58738691796, 101721.04202940996,
       99629.01053730404, 97617.29632020541, 98988.23660304002,
       98061.35894838106, 88974.70416294807, 90420.00960609666,
       75423.09286559872, 89577.70222195369, 69606.52160902979,
       83684.97603990567, 74762.74617647551, 74956.3110490757,
       70575.9937138199, 60100.26821870188, 64585.14721292391,
       47588.36

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(Y, y_pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 78406792.88803764


In [ ]:
from sklearn.metrics import r2_score

r_squared = r2_score(Y, y_pred)
print(f"R-squared: {r_squared}")

R-squared: 0.9507524843355148


In [ ]:
X1 = data_encoded.loc[:,['R&D Spend', 'Administration', 'Marketing Spend']]
Y = data_encoded.loc[:,'Profit']

In [ ]:
regressor = LinearRegression()
regressor.fit(X1,Y)

LinearRegression()

In [ ]:
y_pred1 = regressor.predict(X1)
y_pred1

array([192521.25289008, 189156.76823227, 182147.2790962 , 173696.70002553,
       172139.51418327, 163580.7805712 , 158114.09666865, 160021.36304781,
       151741.69969865, 154884.68410995, 135509.01636714, 135573.71296074,
       129138.05418243, 127487.99166275, 149548.64633453, 146235.1599852 ,
       116915.40540144, 130192.44720781, 129014.2268059 , 115635.21636716,
       116639.6692309 , 117319.45164029, 114706.98171695, 109996.61522126,
       113362.96611314, 102237.72506481, 110600.5753503 , 114408.07145684,
       101660.02600497, 101794.98345176,  99452.37293606,  97687.85627575,
        99001.32898549,  97915.00780465,  89039.27374116,  90511.59956753,
        75286.17458546,  89619.5377079 ,  69697.43064804,  83729.01197692,
        74815.95399105,  74802.55623866,  70620.41182056,  60167.03996335,
        64611.3549157 ,  47650.64968691,  56166.20685261,  46490.58898335,
        49171.38815763,  48215.1341113 ])

In [ ]:
mse = mean_squared_error(Y, y_pred1)
print(f"Mean Squared Error: {mse}")
r_squared = r2_score(Y, y_pred1)
print(f"R-squared: {r_squared}")

Mean Squared Error: 78417126.01913083
R-squared: 0.9507459940683246
